All the fire labels come from the processed VIIRS data set. No need to subset from a different df

In [25]:
import pandas as pd
df_daily_grid_test = {"grid_id": [1,1,1,1,2,2,2,2],
                      "date": pd.to_datetime(["2020-01-12",     # Expect to sample 7 days before this value
                                              "2020-02-10",     # Part of 7 days of fire lbl 2, expect not to sample this value
                                              "2020-03-01",     # Expect to sample
                                              "2020-02-04",     # Expect to exclude as the prior 7 days overlap with fire signal

                                              "2020-01-18",    # Expect not to sample for grid 2 as it overlaps 
                                              "2019-12-28",    # Expect not to sample as it overlaps with 7 days fire    
                                              "2020-02-15",
                                              "2020-03-04"]),
                      "fire_lbl": [0] * 8}

df_viirs_test = {"grid_id" : [1,1,1,1,2,2,2,2],
                 "date"    : pd.to_datetime(["2020-01-20",
                                             "2020-02-15",
                                             "2020-02-20",
                                             "2020-02-01",

                                             "2020-01-20",
                                             "2020-01-01",
                                             "2020-03-15",
                                             "2020-04-04"]),
                 "fire_lbl": [1] * 8}

df_viirs_test      = pd.DataFrame(df_viirs_test)
df_daily_grid_test = pd.DataFrame(df_daily_grid_test)

In [29]:
# Initialise list to store the sampled values
dict_sampled_values = {'date'   : [],
                       'grid_id': []}

# Generate 7 day window for label
window_size = 7 

for i, r in df_viirs_test.iterrows():
    current_date   = r['date']
    current_grid   = r['grid_id']
    current_window = pd.date_range(start = (current_date - pd.Timedelta(days=window_size)), 
                                   end   = current_date)
    
    dict_sampled_values['date'].extend(current_window)
    dict_sampled_values['grid_id'].extend([current_grid] * len(current_window))

                            
df_sampled = pd.DataFrame(dict_sampled_values)
df_sampled = df_sampled.drop_duplicates(subset = ['grid_id','date'])
df_sampled

,date,grid_id
0,2020-01-13,1
1,2020-01-14,1
2,2020-01-15,1
3,2020-01-16,1
4,2020-01-17,1
...,...,...
59,2020-03-31,2
60,2020-04-01,2
61,2020-04-02,2
62,2020-04-03,2


In [ ]:
df_nofire       = df_daily_grid_test[df_daily_grid_test['fire_lbl'] == 0]
sampling_report = []
# Loop over the fire labels again, as these are the ones we need to find matching non fire values for 
for i, r in df_viirs_test.iterrows():
   # Extract date, grid_id
   current_date    = r['date']
   current_grid_id = r['grid_id']
   # Get all the used as fire days for a given grid_id
   #fire_dates = set(df_sampled['date'][df_sampled['grid_id'] == current_grid_id]) # Can be removed as this is covered by last check of nonfire_sample
   
   # Create fire grid id to see which values mapped to which fire grids
   row_group_id        = f"{current_date.strftime('%Y%m%d')}_{current_grid_id}"
   row                 = r[['date','grid_id','fire_lbl']].copy()
   row['fire_grid_id'] = row_group_id
   sampling_report.append(row)

   try:
        nonfire_sample = df_nofire[(
                                    (df_nofire['grid_id'] == current_grid_id) 
                                    and
                                    ((df_nofire['date'] >= (current_date - pd.DateOffset(days=30))) |
                                     (df_nofire['date'] <= (current_date + pd.DateOffset(days=7))))
                                    and
                                   # (df_nofire['date'] not in fire_dates)  # These two lines can be removed as the logic is covered by last check below
                                   # and
                                    # Ensure this is not a value that we have already sampled
                                    (df_nofire[['date', 'id']] not in df_sampled[['date', 'id']] )
                                   )]
        for i_inner, r_inner in nonfire_sample.iterrows():
              day_nofire = r_inner['date']
              nonfire_days.update(pd.date_range(start = (day_nofire - pd.Timedelta(days=7)), end = day_nofire))
              if len(nonfire_days.intersection(fire_dates) > 0):
                   next
              nofire_row =  r_inner[['date','grid_id','fire_lbl']].copy()
              nofire_row['fire_grid_id'] = row_group_id
              sampling_report.append(row)
            
    except ValueError:
        print(f"No Non-Fire samples available for fire label on {row}")

{Timestamp('2020-02-16 00:00:00'), Timestamp('2020-01-25 00:00:00'), Timestamp('2020-01-30 00:00:00'), Timestamp('2020-02-09 00:00:00'), Timestamp('2020-02-12 00:00:00'), Timestamp('2020-02-19 00:00:00'), Timestamp('2020-01-20 00:00:00'), Timestamp('2020-02-14 00:00:00'), Timestamp('2020-01-28 00:00:00'), Timestamp('2020-02-18 00:00:00'), Timestamp('2020-02-20 00:00:00'), Timestamp('2020-01-19 00:00:00'), Timestamp('2020-01-29 00:00:00'), Timestamp('2020-02-17 00:00:00'), Timestamp('2020-01-18 00:00:00'), Timestamp('2020-01-15 00:00:00'), Timestamp('2020-02-08 00:00:00'), Timestamp('2020-02-01 00:00:00'), Timestamp('2020-02-15 00:00:00'), Timestamp('2020-01-31 00:00:00'), Timestamp('2020-01-14 00:00:00'), Timestamp('2020-02-10 00:00:00'), Timestamp('2020-02-11 00:00:00'), Timestamp('2020-01-26 00:00:00'), Timestamp('2020-01-13 00:00:00'), Timestamp('2020-01-17 00:00:00'), Timestamp('2020-02-13 00:00:00'), Timestamp('2020-01-27 00:00:00'), Timestamp('2020-01-16 00:00:00')}
{Timestamp('2